In [ ]:
BAT_CALL_JSON = "BD-A-99-001-analysis.json"
PASS_GAP_MS = 80.0

In [ ]:
%run pathutils.ipynb
%run export.ipynb

# Inter-Pulse Interval (IPI) Profile

This chart shows the time between successive detected pulses, usually plotted
in real milliseconds against pulse order through the recording.

## Purpose:

- To reveal how pulse timing changes across the call sequence
- To distinguish broad behavioural phases such as search, approach, and terminal buzz
- To show whether the sequence becomes more tightly spaced toward the end

## Interpretation

- Larger IPI values generally indicate more widely spaced search-phase calls
- A downward trend suggests the bat is increasing its sampling rate as it approaches a target
- Very short, tightly clustered IPIs are characteristic of a terminal feeding buzz
- Irregularity is not necessarily a problem: real sequences often contain local variation

## Notes

- This is often the clearest single chart for understanding behavioural structure
- Interpretation is strongest when the pulse detection is visually confirmed against the waveform

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


def plot_ipi(df, pass_id, export_folder, export_file_stem):
    plt.figure(figsize=(12, 4))

    plt.plot(df["pulse_index"], df["real_ipi_next_ms"], marker="o")

    # Highlight buzz pulses
    buzz = df[df["is_terminal_buzz"] == True]
    plt.scatter(buzz["pulse_index"], buzz["real_ipi_next_ms"])

    plt.xlabel("Pulse index")
    plt.ylabel("IPI (ms, real time)")
    plt.title(f"Inter-Pulse Interval (IPI) Profile for {BAT_CALL_JSON}, Pass {pass_id}")

    plt.grid(True)
    export_chart(export_folder, f"{export_file_stem}-{str(pass_id).zfill(3)}-ipi", "png")
    plt.show()

# Frequency Profile Across Pulses

This chart shows the estimated frequency of each detected pulse, usually using
one representative value per pulse (for example dominant or mid-pulse frequency).

## Purpose

- To show how call frequency changes across the pulse sequence
- To reveal broad frequency range, within-sequence shifts, and late-stage changes
- To support interpretation of call structure alongside timing information

## Interpretation

- A relatively stable band suggests broadly similar calls through the sequence
- Gradual or abrupt shifts may reflect changes in behavioural phase
- In some species, the final approach or buzz may include a fall in frequency
- Sparse points are normal if frequency could only be estimated for stronger or longer pulses

## Notes
- Frequency estimates for very short or weak pulses may be missing or unstable
- This chart is best read together with the IPI profile rather than on its own
- For time-expansion recordings, values should be interpreted in real frequency if the
expansion factor has already been appliedAA

In [ ]:
import matplotlib.pyplot as plt

def plot_frequency_profile(df, pass_id, export_folder, export_file_stem):
    plt.figure(figsize=(12, 4))

    plt.plot(
        df["pulse_index"],
        df["spectral.dominant_frequency_mid_hz_real"] / 1000,
        marker="o"
    )

    plt.xlabel("Pulse index")
    plt.ylabel("Frequency (kHz, real)")
    plt.title(f"Frequency Profile Across Pulses for {BAT_CALL_JSON}, Pass {pass_id}")

    plt.grid(True)
    export_chart(export_folder, f"{export_file_stem}-{str(pass_id).zfill(3)}-frequency", "png")
    plt.show()

# Pulse Duration

This chart shows the duration of each detected pulse, usually in real milliseconds,
across the sequence of calls in the recording.

## Purpose

- To show how the time-length of pulses varies through the recording
- To identify whether pulses become shorter or otherwise change shape during approach
- To complement timing and frequency plots with a simple measure of pulse structure

## Interpretation

- Longer durations may reflect broader or more extended pulse envelopes
- Shortening pulse durations toward the end of a sequence can be associated with
late-stage approach or feeding buzz behaviour
- Sudden changes may indicate a shift in call type or a change in segmentation

## Notes

- Duration depends on the detection boundaries used by the analyser, so values should
be interpreted as measured pulse regions rather than absolute biological truth
- This chart is especially useful when compared with IPI and buzz labels

In [ ]:
import matplotlib.pyplot as plt

def plot_pulse_duration(df, pass_id, export_folder, export_file_stem):
    plt.figure(figsize=(12, 4))

    plt.plot(df["pulse_index"], df["real_duration_ms"], marker="o")

    plt.xlabel("Pulse index")
    plt.ylabel("Pulse duration (ms, real)")
    plt.title(f"Pulse Duration for {BAT_CALL_JSON}, Pass {pass_id}")

    plt.grid(True)
    export_chart(export_folder, f"{export_file_stem}-{str(pass_id).zfill(3)}-duration", "png")
    plt.show()

# Amplitude Profile

This chart shows a simple amplitude measure for each detected pulse, such as peak
amplitude or RMS amplitude, across the sequence.

## Purpose

- To show how pulse strength varies through the recording
- To help identify stronger and weaker parts of the sequence
- To provide context for missing or less reliable spectral estimates

## Interpretation

- Higher values indicate pulses that are stronger in the recorded signal
- Lower values may reflect weaker calls, greater distance, recording angle, or noise
- Amplitude changes across the sequence can sometimes support interpretation of
approach behaviour, but should be treated cautiously

## Notes

- Recorded amplitude is influenced by many factors besides the bat itself, including
microphone direction, range, background noise, and processing
- This chart is most useful as supporting context rather than as a primary behavioural metric

In [ ]:
import matplotlib.pyplot as plt

def plot_amplitude_profile(df, pass_id, export_folder, export_file_stem):
    plt.figure(figsize=(12, 4))

    plt.plot(df["pulse_index"], df["peak_amplitude"], marker="o")

    plt.xlabel("Pulse index")
    plt.ylabel("Peak amplitude")
    plt.title(f"Amplitude Profile for {BAT_CALL_JSON}, Pass {pass_id}")

    plt.grid(True)
    export_chart(export_folder, f"{export_file_stem}-{str(pass_id).zfill(3)}-amplitude", "png")
    plt.show()

In [ ]:
import json
import pandas as pd
from pathlib import Path

# Construct the full path to the JSON file, which is expected to be in the "data" folder
json_file = (Path("..") / "data" / BAT_CALL_JSON).resolve()

# Load the file and get the list of pulses
with open(json_file, "r") as f:
    data = json.load(f)

# Check this is a time expansion analysis
if data["analysis_mode"] != "time-expansion":
    message = f"Expected analysis_mode='time-expansion', got {data['analysis_mode']!r}. Aborting notebook."
    raise ValueError(message)

pulses = data["pulses"]

# Convert to a DataFrame
df = pd.json_normalize(pulses)

# Add a pulse index number
df["pulse_index"] = range(1, len(df) + 1)

In [ ]:
# Assign pass IDs to each pass in the input file
pass_ids = []
current_pass = 1

for _, row in df.iterrows():
    ipi_prev = row["real_ipi_prev_ms"]

    if pd.notna(ipi_prev) and ipi_prev > PASS_GAP_MS:
        current_pass += 1

    pass_ids.append(current_pass)

df["pass_id"] = pass_ids

In [ ]:
# Generate a summary of the identified passes
pass_summary = (
    df.groupby("pass_id")
      .agg(
          pulse_count=("pulse_index", "count"),
          start_s=("real_start_time_s", "min"),
          end_s=("real_end_time_s", "max"),
          has_buzz=("is_terminal_buzz", "max"),
          min_ipi_ms=("real_ipi_next_ms", "min"),
          mean_freq_khz=("spectral.dominant_frequency_mid_hz_real", lambda s: s.dropna().mean() / 1000 if s.dropna().any() else None),
      )
      .reset_index()
)

In [ ]:
from pathlib import Path

# Construct the export file stemp and folder
export_file_stem = Path(BAT_CALL_JSON).stem
export_folder = Path(get_project_root_folder()) / "exported"
export_folder.mkdir(parents=True, exist_ok=True)

# Initialise the export dictionary
export_dict = {
    "Passes": pass_summary
}

# Iterate over eacg pass
for pass_id, pass_df in df.groupby("pass_id"):
    # Add this pass to the export dictionary
    export_dict[f"Pass {str(pass_id).zfill(3)}"] = pass_df

    # Chart the data
    plot_ipi(pass_df, pass_id, export_folder, export_file_stem)
    plot_frequency_profile(pass_df, pass_id, export_folder, export_file_stem)
    plot_pulse_duration(pass_df, pass_id, export_folder, export_file_stem)
    plot_amplitude_profile(pass_df, pass_id, export_folder, export_file_stem)

# Export the data
export_to_spreadsheet(export_folder, f"{export_file_stem}.xlsx", export_dict)